## Parte 1- Scraping de decretos de emergencia por lluvias

Reunimos los decretos supremos de la PCM publicados entre el 1 de enero y el 31 de mayo de 2025.

In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
import pandas as pd
import requests
from bs4 import BeautifulSoup
import os
import re
import time

**Paso 1: Robots**

In [2]:
respuesta_robots = requests.get(
    "https://www.gob.pe/robots.txt",
    headers={"User-Agent": "Mozilla/5.0"},
    timeout=30,
)
print("Estado:", respuesta_robots.status_code)
print(respuesta_robots.text)

Estado: 200
# See http://www.robotstxt.org/robotstxt.html for documentation on how to use the robots.txt file
#
# To ban all spiders from the entire site uncomment the next two lines:
User-agent: *
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: AhrefsBot
Disallow: /

User-agent: GPTBot
Crawl-delay: 5
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

User-agent: OAI-SearchBot
Crawl-delay: 2
Disallow: /admin/
Disallow: /*?sheet=
Disallow: /*&sheet=

Sitemap: https://www.gob.pe/sitemaps/sitemap.xml.gz



**Qué prohíbe:**
Para todos los robots (`User-agent: *`), el sitio prohíbe el acceso a `/admin/` y a las URL que contienen el parámetro `sheet=`. Para `AhrefsBot` prohíbe todo el sitio (`Disallow: /`). Las líneas que bloquearían a todos los robots están comentadas con `#`, así que no se aplican.

**¿Está permitida /busquedas?**
Sí. La ruta `/busquedas` no aparece en ninguna regla `Disallow` y nuestra URL no contiene el parámetro `sheet=`, así que el robots.txt no prohíbe rastrearla.

**¿Por qué igual hacemos pausas?**
Para los robots en general (`User-agent: *`) no se fija un `Crawl-delay`, pero a otros bots sí se les pide: 5 segundos a GPTBot y 2 segundos a OAI-SearchBot. Ese es el ritmo que el sitio considera razonable. gob.pe es un servicio público: si le mandamos pedidos seguidos lo cargamos sin necesidad y nos pueden bloquear. Por eso esperamos 2 s entre páginas de búsqueda y 1 s entre páginas de normas.


**Paso 2 · Abrir un mes con Selenium**

La página de búsquedas dibuja los resultados con JavaScript: si la pidiéramos con `requests`, llegaría vacía. Por eso usamos Selenium, que abre un Chrome de verdad. Con `WebDriverWait` esperamos hasta que aparezca el primer `<article>` (máximo 30 segundos), en vez de esperar un tiempo fijo con `time.sleep()`: así no seguimos antes de que la página cargue ni esperamos de más.

In [3]:
opciones = Options()
# opciones.add_argument("--headless=new")   # descomenten para no ver la ventana
opciones.add_argument("--window-size=1400,900")

navegador = webdriver.Chrome(options=opciones)

In [4]:
URL_BASE = (
    "https://www.gob.pe/busquedas?contenido[]=normas&institucion[]=pcm"
    "&term=estado%20de%20emergencia%20precipitaciones&sort_by=recent"
)

navegador.get(URL_BASE + "&desde=01-01-2025&hasta=31-01-2025")

# Los resultados los dibuja JavaScript: esperamos a que aparezca el primer <article>
WebDriverWait(navegador, 30).until(
    EC.presence_of_element_located((By.TAG_NAME, "article"))
)
time.sleep(1)   # margen para que terminen de aparecer todos

**Paso 3 · Número Total de resultados en el mes de enero**

La página dice cuántos resultados encontró (12 Resultados). Leemos ese número con una expresión regular para compararlo después con los que realmente extraemos.

In [5]:
texto_pagina = navegador.find_element(By.TAG_NAME, "body").text
total_enero = int(re.search(r"(\d+) Resultados?", texto_pagina).group(1))
print("Resultados que dice la página:", total_enero)

Resultados que dice la página: 12


**Paso 4. De cada resultado en el mes de enero**

Cada resultado es un `<article>`. Primero tomamos todos los `article` y después buscamos cada dato dentro de cada uno: el número es el texto del enlace principal (`a.link-principal`), el enlace es su `href`, la fecha es el texto que empieza con "Publicado:" y el título es el párrafo `<p>`. La fecha la buscamos por lo que dice y no por su posición, para que no falle si cambia el orden. Al final juntamos todo en la función `leer_articulo`, porque la vamos a usar en los cinco meses.

In [6]:
articulos = navegador.find_elements(By.TAG_NAME, "article")
print("Artículos encontrados:", len(articulos))
primero = articulos[0]
print(primero.text)

Artículos encontrados: 12
PCM - Presidencia del Consejo de Ministros
Normas y documentos legales
›
Decretos
Decreto Supremo N.° 018-2025-PCM
Publicado: 30 de enero de 2025
Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto 
Descargar archivo


Numero de la norma

In [7]:
enlace_principal = primero.find_element(By.CSS_SELECTOR, "a.link-principal")
numero = enlace_principal.text     # "Decreto Supremo N.° 0XX-2025-PCM"
print(numero)

Decreto Supremo N.° 018-2025-PCM


Enlace de la norma

In [8]:
enlace = enlace_principal.get_attribute("href")   # Selenium lo devuelve completo, con https://www.gob.pe
print(enlace)

https://www.gob.pe/institucion/pcm/normas-legales/6529837-018-2025-pcm


Fecha de la publicacion

In [9]:
# La fecha se busca por lo que dice ("Publicado:"), no por su posición
fecha = None
for span in primero.find_elements(By.TAG_NAME, "span"):
    if span.text.startswith("Publicado:"):
        fecha = span.text.replace("Publicado:", "").strip()
print(fecha)

30 de enero de 2025


Titulo de la publicacion

In [10]:
titulo = primero.find_element(By.TAG_NAME, "p").text
print(titulo)

Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto 


Resultado de cada articulo

In [11]:
def leer_articulo(articulo):
    """Convierte un resultado de la búsqueda de gob.pe en un diccionario."""
    enlace_principal = articulo.find_element(By.CSS_SELECTOR, "a.link-principal")

    fecha = None
    for span in articulo.find_elements(By.TAG_NAME, "span"):
        if span.text.startswith("Publicado:"):
            fecha = span.text.replace("Publicado:", "").strip()

    parrafos = articulo.find_elements(By.TAG_NAME, "p")
    titulo = parrafos[0].text if parrafos else None

    return {
        "numero": enlace_principal.text,
        "fecha": fecha,
        "titulo": titulo,
        "enlace": enlace_principal.get_attribute("href"),
    }


for articulo in articulos:
    print(leer_articulo(articulo))

{'numero': 'Decreto Supremo N.° 018-2025-PCM', 'fecha': '30 de enero de 2025', 'titulo': 'Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto ', 'enlace': 'https://www.gob.pe/institucion/pcm/normas-legales/6529837-018-2025-pcm'}
{'numero': 'Decreto Supremo N.° 016-2025-PCM', 'fecha': '29 de enero de 2025', 'titulo': 'Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco ', 'enlace': 'https://www.gob.pe/institucion/pcm/normas-legales/6529813-016-2025-pcm'}
{'numero': 'Decreto Supremo N.° 013-2025-PCM', 'fecha': '23 de enero de 2025', 'titulo': 'Decreto Supremo que modifica el Decreto Supremo N° 003-2025-PCM, Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad ', 'enlace': 'https://www.gob.pe/instit

**Paso 5. Para todos los meses, entre el 01 de enero al 31 de mayo 2025**

Repetimos los pasos 2 a 4 para cada mes de la temporada (enero a mayo de 2025), cambiando las fechas `desde` y `hasta` en la URL. En cada mes esperamos con `WebDriverWait` a que carguen los resultados, leemos el total que dice la página y extraemos cada `article` con `leer_articulo`. Volvemos a buscar los `article` en cada página porque los de la página anterior ya no existen. Entre una página y otra esperamos al menos 2 segundos, como explicamos en el paso 1.

In [12]:
meses = [
    ("enero",   "01-01-2025", "31-01-2025"),
    ("febrero", "01-02-2025", "28-02-2025"),
    ("marzo",   "01-03-2025", "31-03-2025"),
    ("abril",   "01-04-2025", "30-04-2025"),
    ("mayo",    "01-05-2025", "31-05-2025"),
]

In [13]:
todas_las_normas = []
verificacion = []

for nombre_mes, desde, hasta in meses:
    navegador.get(URL_BASE + f"&desde={desde}&hasta={hasta}")
    WebDriverWait(navegador, 30).until(
        EC.presence_of_element_located((By.TAG_NAME, "article"))
    )
    time.sleep(1)

    texto_pagina = navegador.find_element(By.TAG_NAME, "body").text
    total = int(re.search(r"(\d+) Resultados?", texto_pagina).group(1))

    # Se vuelven a buscar en cada página: los article de la página anterior ya no existen
    articulos = navegador.find_elements(By.TAG_NAME, "article")
    for articulo in articulos:
        fila = leer_articulo(articulo)
        fila["mes"] = nombre_mes
        todas_las_normas.append(fila)

    verificacion.append({
        "mes": nombre_mes,
        "resultados_totales": total,
        "resultados_extraidos": len(articulos),
    })
    print(nombre_mes, "listo")
    time.sleep(2)   # pausa entre páginas

enero listo
febrero listo
marzo listo
abril listo
mayo listo


In [14]:
navegador.quit()

Mostramos las 54 normas extraídas de enero a mayo: número, fecha, título y enlace de cada resultado, con el mes de la búsqueda.

In [15]:
# Resultado del paso 5: todas las normas extraídas, de enero a mayo
tabla_extraida = pd.DataFrame(todas_las_normas)
print("Normas extraídas:", len(tabla_extraida))
tabla_extraida[["mes", "numero", "fecha", "titulo", "enlace"]]

Normas extraídas: 54


,mes,numero,fecha,titulo,enlace
0,enero,Decreto Supremo N.° 018-2025-PCM,30 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
1,enero,Decreto Supremo N.° 016-2025-PCM,29 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
2,enero,Decreto Supremo N.° 013-2025-PCM,23 de enero de 2025,Decreto Supremo que modifica el Decreto Suprem...,https://www.gob.pe/institucion/pcm/normas-lega...
3,enero,Decreto Supremo N.° 012-2025-PCM,22 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
4,enero,Decreto Supremo N.° 011-2025-PCM,22 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
5,enero,Decreto Supremo N.° 008-2025-PCM,10 de enero de 2025,Declaratoria del Estado de Emergencia,https://www.gob.pe/institucion/pcm/normas-lega...
6,enero,Decreto Supremo N.° 007-2025-PCM,10 de enero de 2025,Declaratoria del Estado de Emergencia,https://www.gob.pe/institucion/pcm/normas-lega...
7,enero,Decreto Supremo N.° 006-2025-PCM,8 de enero de 2025,Decreto Supremo que prorroga el Estado de Eme...,https://www.gob.pe/institucion/pcm/normas-lega...
8,enero,Decreto Supremo N.° 004-2025-PCM,3 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...
9,enero,Decreto Supremo N.° 003-2025-PCM,3 de enero de 2025,Decreto Supremo que prorroga el Estado de Emer...,https://www.gob.pe/institucion/pcm/normas-lega...


**Paso 6. Verificacion de los resultados totales**

En los cinco meses, el número de resultados que dice la página coincide con el que extrajimos (54 en total). Eso significa que cada mes cabía en una sola página y que no se nos escapó ningún resultado.

In [16]:
tabla_verificacion = pd.DataFrame(verificacion)
tabla_verificacion["coinciden"] = (
    tabla_verificacion["resultados_totales"] == tabla_verificacion["resultados_extraidos"]
)
tabla_verificacion

,mes,resultados_totales,resultados_extraidos,coinciden
0,enero,12,12,True
1,febrero,4,4,True
2,marzo,12,12,True
3,abril,15,15,True
4,mayo,11,11,True


**Paso 7. Eliminar normas repetidas**

Juntamos en una sola tabla las normas de los 5 meses. Para detectar repetidas usamos el **enlace**, porque cada norma tiene su propia página en gob.pe: dos normas distintas nunca comparten enlace. Una norma se podría repetir si dos búsquedas tienen fechas en común o si gob.pe muestra dos veces el mismo resultado.

In [17]:
normas = pd.DataFrame(todas_las_normas)
print("Normas extraídas:", len(normas))

normas = normas.drop_duplicates(subset="enlace")
print("Normas sin repetir:", len(normas))
print("Repetidas eliminadas:", len(todas_las_normas) - len(normas))

Normas extraídas: 54
Normas sin repetir: 54
Repetidas eliminadas: 0


En nuestra temporada no hubo repetidas: cada búsqueda cubre un mes distinto, sin días en común. Igual dejamos el paso porque, si se cambian las fechas o gob.pe repite un resultado, el código lo corrige solo.

**Paso 8. Quedarnos solo con las normas de la PCM**

Aunque la búsqueda filtra por PCM, se cuelan normas de otras instituciones. Nos quedamos solo con las que tienen `/institucion/pcm/` en el enlace.

In [18]:
es_de_pcm = normas["enlace"].str.contains("/institucion/pcm/")

# ~ significa "no": las filas que NO son de la PCM
print("Normas de otras instituciones que se quitaron:", (~es_de_pcm).sum())
print(normas.loc[~es_de_pcm, ["mes", "numero", "enlace"]].to_string())

normas = normas[es_de_pcm].reset_index(drop=True)
print("\nNormas de la PCM que quedan:", len(normas))

Normas de otras instituciones que se quitaron: 1
     mes                                     numero                                                                               enlace
51  mayo  Resolución Directoral N.° 121-2025-DG-HSB  https://www.gob.pe/institucion/honadomani-sb/normas-legales/6783538-121-2025-dg-hsb

Normas de la PCM que quedan: 53


Se quitó **1 norma**: la Resolución Directoral N.° 121-2025-DG-HSB del Hospital San Bartolomé (mayo), que no tiene relación con la PCM ni con emergencias por lluvias. Quedan **53 decretos supremos** de la PCM.

**Paso 9. Títulos completos con `requests` y `BeautifulSoup`**

En la lista de resultados, los títulos largos aparecen cortados con "..." y lo que se corta suele ser justo el motivo (*"por impacto de daños a consecuencia de intensas precipitaciones..."*). La página de cada norma ya viene armada desde el servidor (no la dibuja JavaScript), así que no necesitamos Selenium: basta con `requests`, que es más rápido que abrir Chrome.

Dos detalles:
- gob.pe **rechaza** los pedidos que no se identifican como navegador (responde con el código 418). Por eso mandamos el encabezado `User-Agent: Mozilla/5.0`.
- El título está en `<div class="description">`, pero ese mismo div trae al final el recuadro del PDF (*"1 DS N° 011-2025-PCM PDF 852.4 KB Descargar"*). Ese recuadro vive en su propio div (`institution-document__files`), así que lo borramos con `.decompose()` antes de leer el texto. Así el título queda limpio.

In [19]:
encabezados = {"User-Agent": "Mozilla/5.0"}


def leer_titulo_completo(enlace):
    """Descarga la página de una norma y devuelve su título completo."""
    respuesta = requests.get(enlace, headers=encabezados, timeout=30)
    respuesta.raise_for_status()   # si la página falla, que se vea el error y no guardemos un título vacío

    sopa = BeautifulSoup(respuesta.text, "html.parser")
    descripcion = sopa.find("div", class_="description")

    # Quitamos el recuadro del PDF para quedarnos solo con el título
    recuadro_pdf = descripcion.find("div", class_="institution-document__files")
    if recuadro_pdf is not None:
        recuadro_pdf.decompose()

    return descripcion.get_text(" ", strip=True)

In [20]:
# Probamos con la primera norma cuyo título aparece cortado en la lista
cortados = normas["titulo"].str.strip().str.endswith("...")
ejemplo = normas[cortados].iloc[0]

print(ejemplo["numero"])
print("En la lista:", ejemplo["titulo"])
print("Completo:   ", leer_titulo_completo(ejemplo["enlace"]))

Decreto Supremo N.° 011-2025-PCM
En la lista: Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huep ...
Completo:    Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios


In [21]:
titulos_completos = []
for enlace in normas["enlace"]:
    titulos_completos.append(leer_titulo_completo(enlace))
    time.sleep(1)   # pausa de 1 segundo entre páginas

normas["titulo_completo"] = titulos_completos
print("Títulos descargados:", len(titulos_completos), "de", len(normas))
print("Títulos que venían cortados en la lista:", cortados.sum())

Títulos descargados: 53 de 53
Títulos que venían cortados en la lista: 24


In [22]:
# Verificación: los títulos que venían cortados ahora están completos
pd.set_option("display.max_colwidth", None)
normas.loc[cortados, ["numero", "titulo", "titulo_completo"]].head(5)

,numero,titulo,titulo_completo
4,Decreto Supremo N.° 011-2025-PCM,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huep ...","Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios"
13,Decreto Supremo N.° 024-2025-PCM,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...","Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, La Libertad, Lambayeque, Lima, Moquegua, Pasco, Piura, Puno, San Martín, Tacna, Tumbes y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales."
14,Decreto Supremo N.° 022-2025-PCM,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Manitea, Kimbiri y Cielo Punco de la provincia de La Convención del departamento de Cusco y en el distrito de Samuga ...","Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Manitea, Kimbiri y Cielo Punco de la provincia de La Convención del departamento de Cusco y en el distrito de Samugari de la provincia de La Mar del departamento de Ayacucho"
15,Decreto Supremo N.° 021-2025-PCM,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, H ...","Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Madre de Dios, Piura, Puno, San Martín, Tacna y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales"
18,Decreto Supremo N.° 036-2025-PCM,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huep ...","Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios"


**Revisión: títulos que no dicen nada**

Al leer los títulos completos encontramos normas cuyo título en gob.pe es solo *"Declaratoria del Estado de Emergencia"*: no dicen dónde ni por qué. Con un título así, el paso 10 no puede saber si la norma es por lluvias. Las buscamos por su largo (los títulos normales tienen más de 90 caracteres):

In [23]:
titulos_genericos = normas[normas["titulo_completo"].str.len() < 60]
titulos_genericos[["numero", "fecha", "titulo_completo", "enlace"]]

,numero,fecha,titulo_completo,enlace
5,Decreto Supremo N.° 008-2025-PCM,10 de enero de 2025,Declaratoria del Estado de Emergencia,https://www.gob.pe/institucion/pcm/normas-legales/6410307-008-2025-pcm
6,Decreto Supremo N.° 007-2025-PCM,10 de enero de 2025,Declaratoria del Estado de Emergencia,https://www.gob.pe/institucion/pcm/normas-legales/6410247-007-2025-pcm


Abrimos el PDF de cada una (botón **Descargar** en su página) y leímos el título oficial que aparece al inicio del decreto:

- **DS 007-2025-PCM**: declara el Estado de Emergencia en varios distritos de 19 departamentos y el Callao **"por peligro inminente ante intensas precipitaciones pluviales"**. **Es un decreto de lluvias.** Si no lo corregimos, perdemos la primera declaratoria de la temporada y 20 departamentos se quedan con una declaratoria menos.
- **DS 008-2025-PCM**: declara el Estado de Emergencia en tres distritos de Chiclayo **"por peligro inminente ante colapso del sistema de alcantarillado"**. No es de lluvias, pero también corregimos su título para que `tipo` y `motivo` salgan bien.

Reemplazamos el título genérico por el título oficial. En el PDF está todo en mayúsculas; lo escribimos con mayúsculas y tildes normales para que la búsqueda de departamentos del paso 12 (que distingue mayúsculas) lo encuentre.

In [24]:
titulos_oficiales = {
    "Decreto Supremo N.° 007-2025-PCM": (
        "Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas "
        "provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, "
        "Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, "
        "Moquegua, Pasco, Puno, San Martín, Tacna y la Provincia Constitucional del Callao, "
        "por peligro inminente ante intensas precipitaciones pluviales"
    ),
    "Decreto Supremo N.° 008-2025-PCM": (
        "Decreto Supremo que declara el Estado de Emergencia en los distritos de Chiclayo, "
        "José Leonardo Ortiz y La Victoria de la provincia de Chiclayo del departamento de "
        "Lambayeque, por peligro inminente ante colapso del sistema de alcantarillado"
    ),
}

for numero, titulo_oficial in titulos_oficiales.items():
    fila = normas["numero"] == numero
    normas.loc[fila, "titulo_completo"] = titulo_oficial
    print(numero, "-> filas corregidas:", fila.sum())

Decreto Supremo N.° 007-2025-PCM -> filas corregidas: 1
Decreto Supremo N.° 008-2025-PCM -> filas corregidas: 1


**Paso 10. Clasificar cada norma: `es_lluvia`, `tipo` y `motivo`**

Usamos el título completo en minúsculas (`.lower()`) para que "Prórroga" y "prórroga" cuenten igual, y el operador `in` para preguntar si una palabra está dentro del título.

En `tipo` **el orden de las preguntas importa**: los títulos de prórroga dicen *"prorroga el Estado de Emergencia **declara**do..."*, y "declarado" contiene "declara". Por eso primero preguntamos por "prorroga" y solo si no está, por "declara". Al revés, todas las prórrogas saldrían como declaratorias.

In [25]:
def es_de_lluvia(titulo):
    titulo = titulo.lower()
    return "lluvia" in titulo or "precipitaciones" in titulo


def clasificar_tipo(titulo):
    titulo = titulo.lower()
    if "prórroga" in titulo or "prorroga" in titulo:
        return "prorroga"
    elif "declara" in titulo:
        return "declara"
    else:
        return "otro"


def clasificar_motivo(titulo):
    titulo = titulo.lower()
    if "peligro inminente" in titulo:
        return "peligro inminente"
    elif "impacto de daños" in titulo:
        return "impacto de daños"
    else:
        return "otro"


normas["es_lluvia"] = normas["titulo_completo"].apply(es_de_lluvia)
normas["tipo"] = normas["titulo_completo"].apply(clasificar_tipo)
normas["motivo"] = normas["titulo_completo"].apply(clasificar_motivo)

print(normas["es_lluvia"].value_counts(), "\n")
print(normas["tipo"].value_counts(), "\n")
print(normas["motivo"].value_counts())

es_lluvia
False    35
True     18
Name: count, dtype: int64 

tipo
prorroga    37
declara     16
Name: count, dtype: int64 

motivo
otro                 31
impacto de daños     17
peligro inminente     5
Name: count, dtype: int64


In [26]:
# Revisión de las filas que quedaron como "otro"
tipo_otro = normas["tipo"] == "otro"
motivo_otro = normas["motivo"] == "otro"

print("Filas con tipo 'otro':", tipo_otro.sum())
print("Filas con motivo 'otro':", motivo_otro.sum())
print("De esas, ¿cuántas son de lluvias?:", normas.loc[tipo_otro | motivo_otro, "es_lluvia"].sum())

normas.loc[tipo_otro | motivo_otro, ["numero", "es_lluvia", "tipo", "motivo", "titulo_completo"]]

Filas con tipo 'otro': 0
Filas con motivo 'otro': 31
De esas, ¿cuántas son de lluvias?: 0


,numero,es_lluvia,tipo,motivo,titulo_completo
0,Decreto Supremo N.° 018-2025-PCM,False,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en las provincias de Putumayo y Mariscal Ramón Castilla del departamento de Loreto
1,Decreto Supremo N.° 016-2025-PCM,False,prorroga,otro,"Decreto Supremo que prorroga el Estado de Emergencia en algunos distritos y centros poblados de provincias pertenecientes a los departamentos de Ayacucho, Huancavelica, Junín y Cusco"
2,Decreto Supremo N.° 013-2025-PCM,False,prorroga,otro,"Decreto Supremo que modifica el Decreto Supremo N° 003-2025-PCM, Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad"
3,Decreto Supremo N.° 012-2025-PCM,False,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en el Corredor Vial Sur Apurímac-Cusco-Arequipa
4,Decreto Supremo N.° 011-2025-PCM,False,prorroga,otro,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Tambopata, Inambari, Las Piedras y Laberinto de la provincia de Tambopata y en los distritos de Madre de Dios y Huepetuhe de la provincia de Manu del departamento de Madre de Dios"
8,Decreto Supremo N.° 004-2025-PCM,False,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Pataz del departamento de La Libertad
9,Decreto Supremo N.° 003-2025-PCM,False,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Trujillo del departamento de La Libertad
10,Decreto Supremo N.° 002-2025-PCM,False,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Virú del departamento de La Libertad
11,Decreto Supremo N.° 001-2025-PCM,False,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Zarumilla del departamento de Tumbes
12,Decreto Supremo N.° 025-2025-PCM,False,prorroga,otro,Decreto Supremo que prorroga el Estado de Emergencia declarado en la provincia de Zarumilla del departamento de Tumbes


In [27]:
# También revisamos al revés: normas que SÍ dicen su motivo pero no son de lluvias
normas.loc[~motivo_otro & ~normas["es_lluvia"], ["numero", "tipo", "motivo", "titulo_completo"]]

,numero,tipo,motivo,titulo_completo
5,Decreto Supremo N.° 008-2025-PCM,declara,peligro inminente,"Decreto Supremo que declara el Estado de Emergencia en los distritos de Chiclayo, José Leonardo Ortiz y La Victoria de la provincia de Chiclayo del departamento de Lambayeque, por peligro inminente ante colapso del sistema de alcantarillado"
7,Decreto Supremo N.° 006-2025-PCM,prorroga,peligro inminente,Decreto Supremo que prorroga el Estado de Emergencia declarado en varios distritos de las provincias del departamento de Piura por peligro inminente ante déficit hídrico
22,Decreto Supremo N.° 032-2025-PCM,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en el sector comprendido entre el Jr. Jauja, Jr. Áncash, Jr. Huánuco, Prolongación Lucanas, Jr. Santa Rosa y Jr. Huanta del Cercado de Lima de la provincia y departamento de Lima, por impacto de daños a consecuencia de gran incendio urbano"
26,Decreto Supremo N.° 027-2025-PCM,prorroga,peligro inminente,"Decreto Supremo que prorroga el Estado de Emergencia declarado en los distritos de Chiclayo, José Leonardo Ortiz y La Victoria de la provincia de Chiclayo del departamento de Lambayeque, por peligro inminente ante colapso del sistema de alcantarillado"


**¿Están bien clasificadas las filas "otro"?**

- **`tipo`**: ninguna fila quedó como "otro"; todas dicen "declara" o "prorroga". Un caso para tener en cuenta es el DS 013-2025-PCM, que *modifica* la prórroga de Trujillo y quedó como "prorroga" porque su título contiene esa palabra. Como no es de lluvias, no afecta el resultado.
- **`motivo`**: 31 filas quedaron como "otro" y **ninguna es de lluvias**. Son estados de emergencia cuyo título no dice el motivo: Pataz, Trujillo y Virú (La Libertad), Lima Metropolitana y Callao, el VRAEM (distritos de La Mar, La Convención y centros poblados de Ayacucho, Huancavelica, Junín y Cusco), el Corredor Vial Sur, Tambopata y Manu (Madre de Dios), Zarumilla (Tumbes) y Putumayo (Loreto). En los PDF que revisamos (por ejemplo, DS 012, 013, 035 y 046) el decreto se justifica en el **control del orden interno** (delincuencia, minería ilegal, narcotráfico), no en un desastre natural. Están bien como "otro": no son "peligro inminente" ni "impacto de daños" por lluvias.
- **Normas con motivo pero sin lluvia**: DS 006 (peligro inminente ante **déficit hídrico** en Piura, es decir, falta de agua), DS 008 y DS 027 (colapso del **alcantarillado** en Chiclayo) y DS 032 (**gran incendio urbano** en el Cercado de Lima). Está bien que tengan `es_lluvia = False`.
- **Todas las normas de lluvias tienen motivo**: ninguna quedó como "otro".
- El único error que encontramos fue el **DS 007-2025-PCM**: con el título genérico de gob.pe habría quedado como `es_lluvia = False` y motivo "otro". Por eso lo corregimos antes de este paso (ver la revisión del paso 9).

**Paso 11. Quedarnos solo con las normas de lluvias**

In [28]:
decretos_lluvias = normas[normas["es_lluvia"]].reset_index(drop=True)

print("Normas de lluvias:", len(decretos_lluvias))
print(decretos_lluvias["tipo"].value_counts(), "\n")
print(decretos_lluvias["motivo"].value_counts())

decretos_lluvias[["numero", "fecha", "tipo", "motivo", "titulo_completo"]].head(5)

Normas de lluvias: 18
tipo
declara     11
prorroga     7
Name: count, dtype: int64 

motivo
impacto de daños     16
peligro inminente     2
Name: count, dtype: int64


,numero,fecha,tipo,motivo,titulo_completo
0,Decreto Supremo N.° 007-2025-PCM,10 de enero de 2025,declara,peligro inminente,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, Puno, San Martín, Tacna y la Provincia Constitucional del Callao, por peligro inminente ante intensas precipitaciones pluviales"
1,Decreto Supremo N.° 024-2025-PCM,26 de febrero de 2025,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, La Libertad, Lambayeque, Lima, Moquegua, Pasco, Piura, Puno, San Martín, Tacna, Tumbes y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales."
2,Decreto Supremo N.° 021-2025-PCM,19 de febrero de 2025,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Madre de Dios, Piura, Puno, San Martín, Tacna y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales"
3,Decreto Supremo N.° 034-2025-PCM,12 de marzo de 2025,declara,impacto de daños,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lambayeque, Lima, Loreto, Moquegua, Pasco, Piura, San Martín, Tacna y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales"
4,Decreto Supremo N.° 033-2025-PCM,12 de marzo de 2025,declara,peligro inminente,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lambayeque, Lima, Loreto, Moquegua, Pasco, Piura, Puno, San Martín, Tacna y de la Provincia Constitucional del Callao, por peligro inminente ante intensas precipitaciones pluviales"


**Paso 12. ¿Qué departamentos menciona cada decreto?**

Buscamos cada departamento en el **título original** (con mayúsculas y tildes, sin `.lower()`) y **por palabra completa** con `\b` (borde de palabra). Así evitamos dos errores:
- **"Ica" dentro de "Huancavelica"**: en minúsculas, `"ica" in "huancavelica"` es `True`. Con `\b`, "Ica" solo cuenta si aparece como palabra suelta.
- **Contar dos veces el mismo departamento**: preguntamos *¿aparece?* (sí o no) con `re.search`, no *¿cuántas veces aparece?*. Si un título dice "provincia de Ica del departamento de Ica", Ica entra **una sola vez** en la lista de ese decreto.

In [29]:
departamentos = ["Amazonas", "Áncash", "Apurímac", "Arequipa", "Ayacucho",
"Cajamarca", "Callao", "Cusco", "Huancavelica", "Huánuco",
"Ica", "Junín", "La Libertad", "Lambayeque", "Lima", "Loreto",
"Madre de Dios", "Moquegua", "Pasco", "Piura", "Puno",
"San Martín", "Tacna", "Tumbes", "Ucayali"]


def buscar_departamentos(titulo):
    """Devuelve la lista de departamentos que menciona el título, cada uno una sola vez."""
    encontrados = []
    for departamento in departamentos:
        if re.search(r"\b" + departamento + r"\b", titulo):
            encontrados.append(departamento)
    return encontrados


decretos_lluvias["departamentos"] = decretos_lluvias["titulo_completo"].apply(buscar_departamentos)
decretos_lluvias["n_departamentos"] = decretos_lluvias["departamentos"].apply(len)

decretos_lluvias[["numero", "tipo", "n_departamentos", "departamentos"]]

,numero,tipo,n_departamentos,departamentos
0,Decreto Supremo N.° 007-2025-PCM,declara,20,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Callao, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, Puno, San Martín, Tacna]"
1,Decreto Supremo N.° 024-2025-PCM,declara,21,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, La Libertad, Lambayeque, Lima, Moquegua, Pasco, Piura, Puno, San Martín, Tacna, Tumbes, Ucayali]"
2,Decreto Supremo N.° 021-2025-PCM,declara,20,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Madre de Dios, Piura, Puno, San Martín, Tacna, Ucayali]"
3,Decreto Supremo N.° 034-2025-PCM,declara,21,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lambayeque, Lima, Loreto, Moquegua, Pasco, Piura, San Martín, Tacna, Ucayali]"
4,Decreto Supremo N.° 033-2025-PCM,declara,22,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Callao, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lambayeque, Lima, Loreto, Moquegua, Pasco, Piura, Puno, San Martín, Tacna]"
5,Decreto Supremo N.° 026-2025-PCM,declara,18,"[Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Moquegua, Pasco, Piura, Tacna, Tumbes, Ucayali]"
6,Decreto Supremo N.° 057-2025-PCM,declara,1,[Loreto]
7,Decreto Supremo N.° 056-2025-PCM,prorroga,7,"[Áncash, Cusco, Huánuco, La Libertad, Pasco, Piura, Tumbes]"
8,Decreto Supremo N.° 053-2025-PCM,prorroga,6,"[Amazonas, Áncash, Cusco, Piura, Tumbes, Ucayali]"
9,Decreto Supremo N.° 052-2025-PCM,declara,2,"[Cusco, Madre de Dios]"


In [30]:
# Verificación 1: todo decreto de lluvias debe mencionar al menos un departamento
sin_departamento = decretos_lluvias[decretos_lluvias["n_departamentos"] == 0]
print("Decretos de lluvias sin ningún departamento:", len(sin_departamento))

# Verificación 2: si un título escribiera un departamento sin tilde, no lo encontraríamos
sin_tilde = ["Ancash", "Apurimac", "Huanuco", "Junin", "San Martin"]
for nombre in sin_tilde:
    veces = decretos_lluvias["titulo_completo"].str.contains(r"\b" + nombre + r"\b").sum()
    print("Títulos que escriben", nombre, "sin tilde:", veces)

Decretos de lluvias sin ningún departamento: 0
Títulos que escriben Ancash sin tilde: 0
Títulos que escriben Apurimac sin tilde: 0
Títulos que escriben Huanuco sin tilde: 0
Títulos que escriben Junin sin tilde: 0
Títulos que escriben San Martin sin tilde: 0


In [31]:
# Ejemplo de nuestra temporada: el DS 043-2025-PCM menciona Huancavelica, pero NO Ica
ejemplo_ica = decretos_lluvias.loc[
    decretos_lluvias["numero"] == "Decreto Supremo N.° 043-2025-PCM", "titulo_completo"
].iloc[0]
print(ejemplo_ica, "\n")

print('Forma incorrecta: "ica" in titulo.lower() ->', "ica" in ejemplo_ica.lower())
print('Forma correcta:   re.search(r"\\bIca\\b", titulo) ->', re.search(r"\bIca\b", ejemplo_ica) is not None)

Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Junín, Lima, Loreto, Puno, San Martín y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales 

Forma incorrecta: "ica" in titulo.lower() -> True
Forma correcta:   re.search(r"\bIca\b", titulo) -> False


In [32]:
# Ejemplo de nuestra temporada con un departamento nombrado dos veces (DS 035-2025-PCM, no es de lluvias)
ejemplo_doble = normas.loc[normas["numero"] == "Decreto Supremo N.° 035-2025-PCM", "titulo_completo"].iloc[0]
print(ejemplo_doble, "\n")
print("Veces que aparece 'Lima' en el título:", len(re.findall(r"\bLima\b", ejemplo_doble)))
print("Departamentos que contamos:", buscar_departamentos(ejemplo_doble))

# Y el caso de Ica como provincia y como departamento
print("\nCaso Ica:", buscar_departamentos("Estado de Emergencia en la provincia de Ica del departamento de Ica"))

Decreto Supremo que declara el Estado de Emergencia en Lima Metropolitana del departamento de Lima y en la Provincia Constitucional del Callao 

Veces que aparece 'Lima' en el título: 2
Departamentos que contamos: ['Callao', 'Lima']

Caso Ica: ['Ica']


**¿Cómo evitamos contar mal?** Un ejemplo de nuestra temporada es el **DS 043-2025-PCM** (3 de abril de 2025), que declara la emergencia en 14 departamentos, entre ellos **Huancavelica**, pero **no Ica**. Si buscáramos `"ica"` en el título en minúsculas, saldría `True` (por "huancavel**ica**") y le sumaríamos a Ica una declaratoria que no tuvo. Lo mismo pasa con el DS 038-2025-PCM. Con `re.search(r"\bIca\b", titulo)` en el título original sale `False`, que es lo correcto. En cambio, en el DS 024-2025-PCM Ica sí aparece como palabra suelta ("..., Huánuco, Ica, La Libertad...") y se cuenta bien.

Para no contar dos veces, la función devuelve cada departamento **una sola vez por decreto**. En el DS 035-2025-PCM ("Lima Metropolitana del departamento de Lima"), "Lima" aparece dos veces en el título, pero cuenta como un solo departamento. Lo mismo pasaría con "provincia de Ica del departamento de Ica". En nuestros decretos de lluvias no hubo ningún departamento repetido dentro del mismo título, pero el código ya lo cubre.

Ahora contamos, para cada departamento, en cuántas **declaratorias** y en cuántas **prórrogas** por lluvias aparece. Dejamos los 25 departamentos, incluso los que tienen 0, para que en la Parte 3 aparezcan con 0 y no desaparezcan del cruce.

In [33]:
conteo = []
for departamento in departamentos:
    declaratorias = 0
    prorrogas = 0
    for _, decreto in decretos_lluvias.iterrows():
        if departamento in decreto["departamentos"]:
            if decreto["tipo"] == "declara":
                declaratorias = declaratorias + 1
            elif decreto["tipo"] == "prorroga":
                prorrogas = prorrogas + 1
    conteo.append({"departamento": departamento, "declaratorias": declaratorias, "prorrogas": prorrogas})

decretos_por_departamento = pd.DataFrame(conteo)
decretos_por_departamento.sort_values("declaratorias", ascending=False)

,departamento,declaratorias,prorrogas
7,Cusco,9,6
0,Amazonas,8,4
1,Áncash,8,6
4,Ayacucho,8,4
3,Arequipa,8,2
8,Huancavelica,8,2
15,Loreto,8,2
14,Lima,8,3
9,Huánuco,8,5
21,San Martín,7,2


In [34]:
# Verificación: el total de la tabla debe ser igual a la suma de departamentos de cada decreto
es_declara = decretos_lluvias["tipo"] == "declara"
print("Filas (departamentos):", len(decretos_por_departamento))
print("Declaratorias en la tabla:", decretos_por_departamento["declaratorias"].sum(),
      "| departamentos mencionados en las declaratorias:", decretos_lluvias.loc[es_declara, "n_departamentos"].sum())
print("Prórrogas en la tabla:", decretos_por_departamento["prorrogas"].sum(),
      "| departamentos mencionados en las prórrogas:", decretos_lluvias.loc[~es_declara, "n_departamentos"].sum())

Filas (departamentos): 25
Declaratorias en la tabla: 157 | departamentos mencionados en las declaratorias: 157
Prórrogas en la tabla: 71 | departamentos mencionados en las prórrogas: 71


**Paso 13. Guardar los archivos**

En `decretos_lluvias.csv` la columna `titulo` lleva el **título completo** (el de la página de cada norma), no el cortado de la lista. Guardamos con `encoding="utf-8-sig"` para que las tildes se vean bien al abrir el archivo en Excel.

In [35]:
os.makedirs("datos", exist_ok=True)   # crea la carpeta si todavía no existe

columnas = ["numero", "fecha", "titulo_completo", "tipo", "motivo", "enlace"]
archivo_decretos = decretos_lluvias[columnas].rename(columns={"titulo_completo": "titulo"})
archivo_decretos.to_csv("datos/decretos_lluvias.csv", index=False, encoding="utf-8-sig")

decretos_por_departamento.to_csv("datos/decretos_por_departamento.csv", index=False, encoding="utf-8-sig")

In [36]:
# Verificación: volvemos a leer los archivos guardados
revisar_decretos = pd.read_csv("datos/decretos_lluvias.csv")
revisar_departamentos = pd.read_csv("datos/decretos_por_departamento.csv")

print("decretos_lluvias.csv:", revisar_decretos.shape, list(revisar_decretos.columns))
print("decretos_por_departamento.csv:", revisar_departamentos.shape, list(revisar_departamentos.columns))
revisar_decretos.head(3)

decretos_lluvias.csv: (18, 6) ['numero', 'fecha', 'titulo', 'tipo', 'motivo', 'enlace']
decretos_por_departamento.csv: (25, 3) ['departamento', 'declaratorias', 'prorrogas']


,numero,fecha,titulo,tipo,motivo,enlace
0,Decreto Supremo N.° 007-2025-PCM,10 de enero de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Moquegua, Pasco, Puno, San Martín, Tacna y la Provincia Constitucional del Callao, por peligro inminente ante intensas precipitaciones pluviales",declara,peligro inminente,https://www.gob.pe/institucion/pcm/normas-legales/6410247-007-2025-pcm
1,Decreto Supremo N.° 024-2025-PCM,26 de febrero de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, La Libertad, Lambayeque, Lima, Moquegua, Pasco, Piura, Puno, San Martín, Tacna, Tumbes y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales.",declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-legales/6539143-024-2025-pcm
2,Decreto Supremo N.° 021-2025-PCM,19 de febrero de 2025,"Decreto Supremo que declara el Estado de Emergencia en varios distritos de algunas provincias de los departamentos de Amazonas, Áncash, Apurímac, Arequipa, Ayacucho, Cajamarca, Cusco, Huancavelica, Huánuco, Ica, Junín, La Libertad, Lima, Loreto, Madre de Dios, Piura, Puno, San Martín, Tacna y Ucayali, por impacto de daños a consecuencia de intensas precipitaciones pluviales",declara,impacto de daños,https://www.gob.pe/institucion/pcm/normas-legales/6539250-021-2025-pcm


**Resumen de la Parte 1**

| Paso | Normas |
|---|---:|
| Extraídas con Selenium (5 meses) | 54 |
| Repetidas eliminadas | 0 |
| De otras instituciones (quitadas) | 1 |
| Decretos supremos de la PCM | 53 |
| Decretos por lluvias | 18 (11 declaratorias y 7 prórrogas) |

De los 18 decretos por lluvias, 16 fueron por **impacto de daños** y 2 por **peligro inminente** (DS 007 y DS 033). Los archivos `datos/decretos_lluvias.csv` y `datos/decretos_por_departamento.csv` son la entrada de la Parte 3.